In [ ]:
#Using an MLP to detect strikefoot and run corrections on the LSTM's prediction
import json
import pandas as pd
from torch.utils.data import DataLoader, Dataset

In [ ]:
with open("/Users/abhinavarora/Desktop/CadenceCV/ml/data/strikefoot_keypoints.json", "r") as file:
    strikefoot_kpts = json.load(file)

with open("/Users/abhinavarora/Desktop/CadenceCV/ml/data/non_strikefoot_keypoints.json", "r") as file:
    non_strikefoot_kpts = json.load(file)

#Will need to flip the direction (only the x-coordinate)

In [ ]:
print(strikefoot_kpts[0])

In [ ]:
#Restructuring the keypoints to flatten out the dict entries in the json file
def flatten_kpts(kpts):
    res = []
    for frame in kpts:
        d = {}
        d["video"] = frame["video"]
        for (index, k) in enumerate(frame["Keypoints"]):
            d["keypoint " + str(index)] = k
        d["Frame"] = frame["Frame"]
        res.append(d)
    
    return res

strikefoot_kpts = flatten_kpts(strikefoot_kpts)
non_strikefoot_kpts = flatten_kpts(non_strikefoot_kpts)

print(strikefoot_kpts)

In [ ]:
strikefoot_df = pd.DataFrame(strikefoot_kpts)
non_strikefoot_df = pd.DataFrame(non_strikefoot_kpts)

non_strikefoot_labels = pd.Series([0] * len(non_strikefoot_df), name="Label")
strikefoot_labels = pd.Series([1] * len(strikefoot_df), name="Label")
strikefoot_df = pd.concat([strikefoot_df,strikefoot_labels], axis=1)
non_strikefoot_df = pd.concat([non_strikefoot_df, non_strikefoot_labels], axis=1)

non_strikefoot_df = non_strikefoot_df.dropna(axis=0, how="any").reset_index(drop=True)
strikefoot_df = strikefoot_df.dropna(axis=0, how="any").reset_index(drop=True)

#Dropping the frames of non strikefoot df that are in strikefoot df provided they have the same video
mask = ~((non_strikefoot_df["Frame"].isin(strikefoot_df["Frame"])) & (non_strikefoot_df["video"].isin(strikefoot_df["video"])))
non_strikefoot_df = non_strikefoot_df[mask]
non_strikefoot_df.reset_index(drop=True)
strikefoot_df.reset_index(drop=True)

#Sorting by frame and side (but for the same videos)
def sort_by_frame_and_side(df:pd.DataFrame):
    vids = df["video"].unique()
    dfs = []
    for v in vids:
        d:pd.Series = df[df["video"] == v]
        d = d.sort_values(["Frame"], axis=0)
        dfs.append(d)
    
    res = pd.DataFrame(pd.concat(dfs, axis=0)).reset_index(drop=True)
    return res

non_strikefoot_df = sort_by_frame_and_side(non_strikefoot_df)
strikefoot_df = sort_by_frame_and_side(strikefoot_df)

In [ ]:
#Runners in videos are facing 2 different directions: left and right both. This means that the normalised bbox keypoint coordinates are changed
#and therefore duplicating our data to show flipped versions of the strikefoot will help the MLP understand strikefoot regardless of what direction
#a label is facing (the strikefoot/not a strikefoot label is invariant of direction). 2 things will be required here: flip idx and changing each x coordinate to 1 - x coordinate
flip_idx = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15, 18, 17, 20, 19]
def mirror_dfs(df:pd.DataFrame):
    #Due to there being 21 keypoints
    for i in range(21):
        for index, row in df.iterrows():
            val = row["keypoint " + str(i)][0]
            row["keypoint " + str(i)][0] = 1 - val

    #Now, swapping the columns based on flip idx:
    for i in range(21):
        temp = df["keypoint " + str(i)]
        df["keypoint " + str(i)] = df["keypoint " + str(flip_idx[i])]
        df["keypoint " + str(flip_idx[i])] = temp

flipped_strikefoot_df = mirror_dfs(strikefoot_df)
flipped_non_strikefoot_df = mirror_dfs(non_strikefoot_df)

In [10]:
print(strikefoot_df.keys())

Index(['video', 'keypoint 0', 'keypoint 1', 'keypoint 2', 'keypoint 3',
       'keypoint 4', 'keypoint 5', 'keypoint 6', 'keypoint 7', 'keypoint 8',
       'keypoint 9', 'keypoint 10', 'keypoint 11', 'keypoint 12',
       'keypoint 13', 'keypoint 14', 'keypoint 15', 'keypoint 16',
       'keypoint 17', 'keypoint 18', 'keypoint 19', 'keypoint 20', 'Frame',
       'Label'],
      dtype='object')


In [ ]:
class MLPDataset(Dataset):
    def __init__(self, features, labels):
        super().__init__()
        self.features = features
        self.labels = labels

    def __getitem__(self, index):
        return super().__getitem__(index), self.features[index], self.labels[index]
    
    def __len__(self):
        return len(self.features)
